# 第 15 章 · 时序差分 TD：走一步就更新，拿"下一步的估计"当目标

本章回答：能不能不等一局结束就更新价值？"用一个估计去更新另一个估计"靠得住吗？TD 和 MC 谁更好？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/15-td.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)  # 打印保留 3 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| E 走出右边 | `td_update(V, 4, 1.0, None, 0.1)` | δ = 0.50，V(E) 0.5 → 0.55 |
| D → E | `td_update(V, 3, 0.0, 4, 0.1)` | δ = 0.050，V(D) 0.5 → 0.505 |
| 比赛 | `compare()` | TD 0.057，MC 0.076 |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `mc_episode_update`**（来自 `ch14_monte_carlo.py`）：first-visit：每个状态只用它第一次出现之后的回报。alpha=None 时用真平均（1/N）。

In [ ]:
def mc_episode_update(V, episode, alpha=None, counts=None, gamma=1.0):
    """first-visit：每个状态只用它第一次出现之后的回报。alpha=None 时用真平均（1/N）。"""
    G, returns = 0.0, {}
    for s, r, _ in reversed(episode):
        G = r + gamma * G
        returns[s] = G  # 倒着走，最后留下的就是第一次出现时的回报
    for s, G in returns.items():
        if alpha is None:
            counts[s] += 1
            V[s] += (G - V[s]) / counts[s]
        else:
            V[s] += alpha * (G - V[s])
    return V

**函数 `random_walk_episode`**（来自 `rl_envs.py`）：从 C 出发，每步等概率往左或往右；走出左边奖励 0，走出右边奖励 +1。

In [ ]:
def random_walk_episode(rng):
    """从 C 出发，每步等概率往左或往右；走出左边奖励 0，走出右边奖励 +1。
    返回 [(状态, 奖励, 下一个状态或 None)]，状态 0–4 对应 A–E。"""
    s, steps = 2, []
    while True:
        ns = s + (1 if rng.random() < 0.5 else -1)
        if ns < 0:
            steps.append((s, 0.0, None))
            return steps
        if ns > 4:
            steps.append((s, 1.0, None))
            return steps
        steps.append((s, 0.0, ns))
        s = ns

**函数 `mc_prediction`**（来自 `ch14_monte_carlo.py`）：下面要用到的小工具。

In [ ]:
def mc_prediction(n_episodes, alpha=None, seed=0, init=0.5):
    rng = np.random.default_rng(seed)
    V, N = np.full(5, init), np.zeros(5)
    for _ in range(n_episodes):
        mc_episode_update(V, random_walk_episode(rng), alpha, N)
    return V

**常量 `WALK_TRUE`**（来自 `rl_envs.py`）

In [ ]:
WALK_TRUE = np.array([1, 2, 3, 4, 5]) / 6

**函数 `rms`**（来自 `ch14_monte_carlo.py`）：下面要用到的小工具。

In [ ]:
def rms(V):
    return float(np.sqrt(np.mean((V - WALK_TRUE) ** 2)))

**函数 `td_update`**（来自 `ch15_td.py`）：下面要用到的小工具。

In [ ]:
def td_update(V, s, r, s_next, alpha, gamma=1.0):
    target = r + (0.0 if s_next is None else gamma * V[s_next])
    delta = target - V[s]
    V[s] += alpha * delta
    return delta

**函数 `td_prediction`**（来自 `ch15_td.py`）：下面要用到的小工具。

In [ ]:
def td_prediction(n_episodes, alpha=0.1, seed=0, init=0.5):
    rng = np.random.default_rng(seed)
    V = np.full(5, init)
    for _ in range(n_episodes):
        for s, r, ns in random_walk_episode(rng):
            td_update(V, s, r, ns, alpha)
    return V

**函数 `compare`**（来自 `ch15_td.py`）：下面要用到的小工具。

In [ ]:
def compare(n_episodes=100, runs=100):
    td = np.mean([rms(td_prediction(n_episodes, 0.1, seed=k)) for k in range(runs)])
    mc = np.mean([rms(mc_prediction(n_episodes, 0.02, seed=k)) for k in range(runs)])
    return td, mc

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
V = np.full(5, 0.5)

In [ ]:
d = td_update(V, 4, 1.0, None, 0.1)

In [ ]:
print(f"E 走出右边：δ = {d:.2f}，V(E) 0.5 → {V[4]:.2f}")

In [ ]:
d = td_update(V, 3, 0.0, 4, 0.1)

In [ ]:
print(f"D → E：δ = {d:.3f}，V(D) 0.5 → {V[3]:.3f}")

In [ ]:
td, mc = compare()

In [ ]:
print(f"100 局后平均 RMS 误差：TD(α=0.1) {td:.3f}，MC(α=0.02) {mc:.3f}")

## 3. 代码里才看得到的

MC 的代码（第 14 章）要先 `random_walk_episode` 拿到整局再更新；TD 的循环 `for s, r, ns in episode: td_update(...)` 每一步都能立刻更新。真正在线学习时，TD 连整局都不用存，这就是它在长任务里的优势。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. 换步长跑 100 局（100 次实验平均）：α = 0.05 → 误差 0.036；0.1 → 0.057；0.3 → 0.111。步子越大，每局后跳得越凶，停不稳。
2. 把 `compare(n_episodes=...)` 改成 10 和 300，看 TD 和 MC 谁先领先。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 周二的预报修正周一的，不等周六；
> TD 目标：一步奖励 + γ × 下一个的估计；方括号是 δ；
> 用估计更新估计叫自举：TD = MC 的采样 + DP 的自举；
> TD 方差小有偏，MC 无偏方差大。